# Tagging Automation with Google Translate (v2)

This notebook:

- Loads an Excel or CSV input file (`Cleaning_Try.xlsx` or similar)
- Auto-detects Google Cloud credential JSON in Downloads or the notebook folder
- Uses Google Translate API when available and falls back to `deep_translator` if not
- Performs flexible relevancy tagging (including fuzzy matching)
- Applies enhanced sentiment retagging with extended negative keywords
- Exports the final tagged data to `Tagged_Output_v2.xlsx`

**Run cells top-to-bottom.**

In [ ]:
# Install required packages (run this cell once)
import sys
!{sys.executable} -m pip install --quiet google-cloud-translate deep-translator pandas openpyxl rapidfuzz xlrd
print('Installed packages (if not already present).')

In [ ]:
# Imports and credential auto-detection
import os, glob, sys, getpass
import pandas as pd, numpy as np, re, json
from rapidfuzz import fuzz, process

# Try to find a Google credentials JSON automatically in Downloads or current folder
def find_google_creds():
    candidates = []
    # common downloads path for current user
    try:
        home = os.path.expanduser('~')
    except Exception:
        home = '.'
    download_paths = [os.path.join(home, 'Downloads'), os.getcwd()]
    for p in download_paths:
        if not os.path.isdir(p):
            continue
        pattern = os.path.join(p, '*.json')
        for f in glob.glob(pattern):
            name = os.path.basename(f).lower()
            if 'translate' in name or 'google' in name or 'credential' in name or 'key' in name:
                candidates.append(f)
    return candidates

creds = find_google_creds()
google_available = False
google_creds_path = None

if creds:
    # prefer first candidate
    google_creds_path = creds[0]
    os.environ['GOOGLE_APPLICATION_CREDENTIALS'] = google_creds_path
    print('Found credentials:', google_creds_path)
else:
    print('No Google credential file auto-detected in Downloads or notebook folder.')

# Initialize translator (Google) if possible; otherwise fallback to deep_translator
use_google = False
translate_client = None
try:
    from google.cloud import translate_v2 as translate
    # initialize client; will raise if credentials invalid/missing
    translate_client = translate.Client()
    use_google = True
    google_available = True
    print('Google Translate client initialized and will be used.')
except Exception as e:
    print('Google Translate not available or failed to initialize:', str(e))
    print('Falling back to deep_translator for translation.')
    google_available = False

# deep-translator fallback
from deep_translator import GoogleTranslator
print('DeepTranslator available as fallback.')

In [ ]:
# --- Load input file (Excel or CSV) ---
# Edit this path to your input file if needed.
input_path = 'Cleaning_Try.xlsx'  # change if your file has a different name or location
sheet_name = 'Sheet1'  # change if needed for Excel files

if not os.path.exists(input_path):
    raise FileNotFoundError(f"Input file not found at: {input_path}. Please upload or change the path and re-run.")

# Read Excel or CSV
if input_path.lower().endswith(('.xls', '.xlsx')):
    df = pd.read_excel(input_path, sheet_name=sheet_name)
else:
    # try flexible CSV reading
    encodings = ['utf-8','latin1','cp1252']
    for enc in encodings:
        try:
            df = pd.read_csv(input_path, encoding=enc, sep=None, engine='python', on_bad_lines='skip')
            break
        except Exception as e:
            last_exc = e
    else:
        raise last_exc

# Clean/normalize column names
df.columns = df.columns.str.strip().str.replace(r'\s+', ' ', regex=True).str.title()
print('Loaded dataset with shape:', df.shape)
print('Columns:', df.columns.tolist())

# Show top 5 rows
df.head()

In [ ]:
# Translation helper function (uses Google if available, otherwise deep_translator)
from html import unescape
def translate_text_auto(text, target='en'):
    if text is None:
        return ''
    txt = str(text)
    if txt.strip() == '':
        return ''
    # If text appears to be English already, skip (simple heuristic: many ascii letters and common English words)
    sample = txt[:200].lower()
    # quick English detection heuristic: presence of common English stopwords or ASCII ratio
    eng_indicators = [' the ', ' and ', ' to ', ' of ', ' is ', ' in ', ' for ']
    ascii_ratio = sum(1 for c in sample if ord(c) < 128) / max(1, len(sample))
    if any(w in sample for w in eng_indicators) and ascii_ratio > 0.8:
        return txt  # likely English

    # Try Google Translate if available
    if google_available:
        try:
            res = translate_client.translate(txt, target_language=target)
            translated = res.get('translatedText', txt)
            # Google returns HTML-escaped text sometimes; unescape
            return unescape(translated)
        except Exception as e:
            # fallback to deep_translator below
            pass

    # Fallback: deep_translator GoogleTranslator (works offline-ish but needs internet)
    try:
        return GoogleTranslator(source='auto', target=target).translate(txt)
    except Exception:
        return txt  # if all fails, return original

In [ ]:
# Relevancy tagging: flexible exact and fuzzy matching
from rapidfuzz import process, fuzz

def is_relevant_to_brand(input_name, text, fuzzy_threshold=85):
    if not input_name or str(input_name).strip() == '':
        return False
    name = str(input_name).lower().strip()
    text_lower = str(text).lower()
    # exact substring
    if name in text_lower:
        return True
    # token-based exact match using word boundaries
    if re.search(r'\b' + re.escape(name) + r'\b', text_lower):
        return True
    # fuzzy match: extract words/phrases in text similar to input_name
    # we compare against sliding windows of words in the text
    tokens = text_lower.split()
    if len(tokens) == 0:
        return False
    # build candidate phrases up to length of name tokens + 2
    name_tokens = name.split()
    max_len = min(len(tokens), len(name_tokens)+2)
    candidates = [' '.join(tokens[i:i+L]) for L in range(1, max_len+1) for i in range(0, len(tokens)-L+1)]
    match = process.extractOne(name, candidates, scorer=fuzz.token_set_ratio)
    if match and match[1] >= fuzzy_threshold:
        return True
    return False

# Apply relevancy tagging on dataframe
def compute_relevancy_row(row):
    input_name = row.get('Input Name', '')
    headline = row.get('Headline', '') or ''
    opening = row.get('Opening Text', '') or ''
    # Translate headline and opening if needed
    headline_en = translate_text_auto(headline)
    opening_en = translate_text_auto(opening)
    full_text = f"{headline_en} {opening_en}"
    return 'Relevant' if is_relevant_to_brand(input_name, full_text) else 'Not Relevant'

# Avoid duplicate column insertions
if 'Relevancy' in df.columns:
    print('Relevancy column exists, it will be overwritten.')
    df.drop(columns=['Relevancy'], inplace=True)
df.insert(0, 'Relevancy', df.apply(compute_relevancy_row, axis=1))
print('Relevancy tagging done.')

In [ ]:
# Sentiment retagging with expanded negative keywords and robust logic
NEGATIVE_KEYWORDS = set([
    'inflation','cautious','debt','lower','loss','down','fell','downgrade',
    'cuts','reduce','drop','warn','slash','mixed','reduction','fed cut',
    'fed rate cut','rate cut','insufficient','cut','weak','uncertain','mute',
    'pressur','fluctuate','diverge','subpoena','blocks','plummeted','fined',
    'costly','overweight','pullback','crash','amid','concern','dispute','pressurise','pressurised','pressuring'
])

POSITIVE_KEYWORDS = set([
    'appointment','promotion','joins as','new ceo','hired','partnership',
    'award','expanding','growth','invest','funding','launched','wins','secured','recognized','acquisition','profit','gain','rise','increase','upgrade'
])

STOCK_REGEX = re.compile(r'stock price (drop|decline|fall|decrease)|share price (drop|decline|fall|decrease)|stock (fell|dipped|slid|tumbled)', re.I)

def text_has_keyword(text, keywords):
    text = str(text).lower()
    for kw in keywords:
        if kw in text:
            return True
    return False

def retag_sentiment_row(row):
    original = str(row.get('Sentiment','')).strip()
    headline = row.get('Headline','') or ''
    opening = row.get('Opening Text','') or ''
    # Translate for analysis
    headline_en = translate_text_auto(headline)
    opening_en = translate_text_auto(opening)
    text = f"{headline_en} {opening_en}".lower()

    # If original is positive, keep
    if original.lower() == 'positive':
        return 'Positive'

    # If positive keywords present => Positive
    if text_has_keyword(text, POSITIVE_KEYWORDS):
        return 'Positive'

    # If stock movement context => Neutral
    if STOCK_REGEX.search(text):
        return 'Neutral'

    # If any negative keyword => Negative
    if text_has_keyword(text, NEGATIVE_KEYWORDS):
        return 'Negative'

    # If contains both strong positive and negative, mark as Neutral/Mixed
    if text_has_keyword(text, POSITIVE_KEYWORDS) and text_has_keyword(text, NEGATIVE_KEYWORDS):
        return 'Neutral'

    # Default fallback: Neutral
    return 'Neutral'

# Apply sentiment retagging
df['New_Sentiment'] = df.apply(retag_sentiment_row, axis=1)
print('Sentiment retagging complete.')

In [ ]:
# Export results to Excel
output_file = 'Tagged_Output_v2.xlsx'
df.to_excel(output_file, index=False)
print('Exported tagged file to:', output_file)
print('\nSentiment counts:\n', df['New_Sentiment'].value_counts(dropna=False))
print('\nRelevancy counts:\n', df['Relevancy'].value_counts(dropna=False))